In [2]:
import os
import shutil
from pathlib import Path

from pyvcell.vcml import Simulation, VcmlReader
from pyvcell.vcml.vcml_simulation import VcmlSpatialSimulation as Solver


In [3]:
# ----- make a workspace
workspace_dir = Path(os.getcwd()) / "workspace"
sim1_dir = workspace_dir / "sim1_dir"
if sim1_dir.exists():
    shutil.rmtree(sim1_dir)
sim2_dir = workspace_dir / "sim2_dir"
if sim2_dir.exists():
    shutil.rmtree(sim2_dir)

# ---- read in VCML file
model_fp = Path(os.getcwd()).parent / "models" / "SmallSpatialProject_3D.vcml"
bio_model1 = VcmlReader.biomodel_from_file(model_fp)

# ---- get the application and the species mappings for species "s0" and "s1"
app = bio_model1.applications[0]
s1_mapping = next(s for s in app.species_mappings if s.species_name == "s1")
s0_mapping = next(s for s in app.species_mappings if s.species_name == "s0")

# ---- add a simulation to the first application in the biomodel (didn't already have a simulation in the VCML file)
new_sim = Simulation(name="new_sim", duration=10.0, output_time_step=0.1, mesh_size=(20, 20, 20))
app.simulations.append(new_sim)

# ---- set the initial concentration of species "s0" and "s1" in the first application
s0_mapping.init_conc = "3+sin(x)+cos(y)+sin(z)"
s1_mapping.init_conc = "3+sin(x+y+z)"

# ---- run simulation, store in sim1_dir, and plot results
# >>>>> This forms the data for the "Field Data" identified by 'sim1_dir' <<<<<<
sim1_result = Solver(bio_model=bio_model1, out_dir=sim1_dir).run(new_sim.name)
# print([c.label for c in sim1_result.channel_data])
# print(sim1_result.time_points[::11])
# sim1_result.plotter.plot_slice_3d(time_index=0, channel_id="s0")
# sim1_result.plotter.plot_slice_3d(time_index=0, channel_id="s1")
# sim1_result.plotter.plot_concentrations()

2025-04-11T20:46:29.058834Z main WARN The use of package scanning to locate Log4j plugins is deprecated.
Please remove the `packages` attribute from your configuration file.
See https://logging.apache.org/log4j/2.x/faq.html#package-scanning for details.
2025-04-11T20:46:29.063381Z main WARN The Logger cbit.vcell.model.Kinetics was created with the message factory org.apache.logging.log4j.message.ReusableMessageFactory@48450643 and is now requested with a null message factory (defaults to org.apache.logging.log4j.message.ParameterizedMessageFactory), which may create log events with unexpected formatting.
2025-04-11T20:46:29.272559Z main WARN The Logger cbit.vcell.mapping.AbstractMathMapping was created with the message factory org.apache.logging.log4j.message.ReusableMessageFactory@48450643 and is now requested with a null message factory (defaults to org.apache.logging.log4j.message.ParameterizedMessageFactory), which may create log events with unexpected formatting.
2025-04-11 16:46:

Simulation Complete in Main() ... 


initializing mesh
numVolume=8000

CartesianMesh::computeNormalsFromNeighbors(), compute normals from neighbors
Membrane Elements -> N=1080
--------Num of points that have zero neighbors 0
--------Num Neighbors before symmetrize 6408
--------Num Neighbors after symmetrize 6888
Total volume=200.1373488
Total FluxArea =0
Total FluxAreaXM =0
Total FluxAreaXP =0
Total FluxAreaYM =0
Total FluxAreaYP =0
Total FluxAreaZM =0
Total FluxAreaZP =0
mesh initialized
preprocessing finished
pdeCount=4, odeCount=0
error opening log file </Users/jimschaff/Documents/workspace/pyvcell/examples/notebooks/workspace/sim1_dir/SimID_433191748_0_.log>
simulation [SimID_433191748_0_] started
temporary directory used is /var/folders/zz/gcfcvgtd5v1cdgj2sw4bjzdr0000gr/T/
sim file name is /var/folders/zz/gcfcvgtd5v1cdgj2sw4bjzdr0000gr/T/SimID_433191748_0_0000.sim
**This is a little endian machine.**
[[[data:0]]]
numVolRegions=2
Region 0: size=6144, offset=0
Region 1: size=1856, offset=6144
# of active points = 8000


In [5]:
import pyvcell.sim_results.widget as widget
from trame.app.jupyter import show

app = widget.App(sim1_result.vtk_data)
layout = await app.run()
show(layout, open_browser=False)

# vtk_data1 = sim1_result.vtk_data
#
# app = App(vtk_data1)
# # asyncio.get_event_loop().run_until_complete(app.run())
# await app.run()

In [ ]:


# # ----- use field data from sim1_dir to set initial concentration of species "s0"
# s0_mapping.init_conc = "vcField('sim1_dir','s0',0.0,'Volume') * vcField('sim1_dir','s1',0.0,'Volume')"
# s1_mapping.init_conc = "5.0"
# # ---- re-run simulation and store in sim2_dir
# # note that the solution of s0 draws from the data from sim1_dir
# sim2_result = Solver(bio_model=bio_model1, out_dir=sim2_dir).run(new_sim.name)
# # sim2_result.plotter.plot_slice_3d(time_index=0, channel_id="s0")
# # sim2_result.plotter.plot_slice_3d(time_index=0, channel_id="s1")
# # sim2_result.plotter.plot_concentrations()
